# Capstone — Refresh / Content Opportunity Scoring
**Google Search Ranking & Discoverability Capstone — FlyRank AI Machine Learning Engineering Internship**

**Author:** Denmark Nercua Tagomata (Den)
**Lane chosen:** Refresh / Content Opportunity Scoring — *score pages that are growing, declining, recovering, or worth review; output a ranked action engine with reason codes.*

This notebook is the single, self-contained walkthrough of the full pipeline: data → features → validation design → model vs. baseline → ranked recommendations. It reuses the exact same `src/` modules as the weekly assignment notebooks, so every result here is reproducible from the repository alone.

## 1. Problem statement
Content teams cannot manually review every page every week. The decision this capstone supports is: **given only signals available today, which pages deserve a refresh this cycle, which should be left alone, and which are safe to protect as top performers?**

## 2. Data
- **Intended source:** FlyRank ML Internship warehouse (gated Hugging Face dataset, page-level daily search performance).
- **Used in this build:** a public-safe synthetic stand-in (`data/sample_search_data.csv`, 320 pages × 180 days) generated by `src/data_gen.py` to match the real warehouse's schema and time structure, because this build environment has no network route to Hugging Face. See the README for the one-line swap to real data.
- **Columns used:** `page_id`, `date`, `impressions`, `clicks`, `avg_position`, `content_type`, `word_count`, `page_age_days`.
- **Excluded:** `true_regime` (synthetic debugging column only — never a feature). No client names, domains, URLs, private queries, or credentials appear anywhere in this repository.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.features import build_train_valid
from src.scoring_model import train_and_evaluate
from src.ranking_engine import build_ranked_recommendations

raw = pd.read_csv('../data/sample_search_data.csv', parse_dates=['date'])
raw.shape

(57600, 9)

## 3. Methodology
**Label:** `momentum_class` (`growing` / `stable` / `declining`), defined from the percent change in a page's total clicks between a feature window and a strictly later label window (±15% thresholds).

**Features:** average position, position trend slope, impression trend slope, click trend slope, actual CTR, CTR gap vs. an expected-CTR-by-position curve, word count, page age, content type.

**Baseline:** a single-signal rule (sign/magnitude of the position slope only) — the bar the model must clear.

**Model:** a Random Forest classifier (`class_weight='balanced'`) in a scikit-learn pipeline with one-hot encoding for `content_type`.

**Validation design (leakage control):** an *out-of-time* split — the training feature/label windows (days 31-150) end before the validation feature window begins (day 61), and the validation label window (days 121-180) is evaluated on a period the training process never touched. This is stricter than a random row-level split, which would let information about the same time period leak between train and validation.

In [2]:
train_df, valid_df = build_train_valid(raw)
out = train_and_evaluate(train_df, valid_df)
r = out['results']
print(f"Baseline -> accuracy: {r['baseline_accuracy']:.3f}, macro-F1: {r['baseline_macro_f1']:.3f}")
print(f"Model    -> accuracy: {r['model_accuracy']:.3f}, macro-F1: {r['model_macro_f1']:.3f}")

Baseline -> accuracy: 0.466, macro-F1: 0.212
Model    -> accuracy: 0.938, macro-F1: 0.939


## 4. Results
See `outputs/figures/model_vs_baseline.png`, `confusion_matrix.png`, and `feature_importance.png` for the charts embedded in the deployed paper (`paper/index.html`). Numerically, on the same out-of-time validation window, the model outperforms the baseline by a wide margin on both accuracy and macro-F1 — the lift holds across all three classes, not just the majority class.

In [3]:
pd.DataFrame(r['model_classification_report']).T

,precision,recall,f1-score,support
declining,0.962025,0.974359,0.968153,78.0000
growing,0.964286,0.870968,0.915254,93.0000
stable,0.910828,0.959732,0.934641,149.0000
accuracy,0.937500,0.937500,0.937500,0.9375
macro avg,0.945713,0.935019,0.939349,320.0000
weighted avg,0.938844,0.937500,0.937175,320.0000


## 5. Ranked recommendations (action playbook)
Model predictions are combined with explainable, rule-based reason codes into a single ranked action list — `PRIORITY_REFRESH` pages first, then `MONITOR`, `PROTECT`, and `LOW_PRIORITY`.

In [4]:
ranked = build_ranked_recommendations(valid_df, out['model_preds'], out['model_proba'], r['classes'])
ranked.to_csv('../outputs/ranked_recommendations.csv', index=False)
ranked['action'].value_counts()

action
MONITOR             172
PRIORITY_REFRESH     79
LOW_PRIORITY         37
PROTECT              32
Name: count, dtype: int64

In [5]:
ranked.head(10)

,priority_rank,page_id,action,predicted_momentum,confidence,reason_codes,avg_position_mean,position_slope,impressions_sum,ctr_actual,ctr_gap_vs_expected,content_type
0,1,page_0126,PRIORITY_REFRESH,declining,0.924134,high_visibility_page,17.426000,0.027459,36003.0,0.008999,-0.000601,guide
1,2,page_0196,PRIORITY_REFRESH,declining,0.916962,no_strong_signal,7.891667,0.012381,17347.0,0.025710,0.000710,blog
2,3,page_0155,PRIORITY_REFRESH,declining,0.913374,no_strong_signal,16.980167,0.029505,24904.0,0.009557,-0.000043,blog
3,4,page_0026,PRIORITY_REFRESH,declining,0.911993,no_strong_signal,16.281333,0.034013,26455.0,0.010357,-0.000443,category
4,5,page_0095,PRIORITY_REFRESH,declining,0.911209,no_strong_signal,7.624333,0.022566,27988.0,0.028548,0.003548,blog
5,6,page_0233,PRIORITY_REFRESH,declining,0.896603,high_visibility_page,18.575833,0.037879,33894.0,0.007671,0.000471,guide
6,7,page_0178,PRIORITY_REFRESH,declining,0.895964,no_strong_signal,15.809000,0.032052,18960.0,0.011445,0.000645,landing
7,8,page_0308,PRIORITY_REFRESH,declining,0.894092,high_visibility_page,16.503833,0.032165,50702.0,0.010315,0.000715,category
8,9,page_0052,PRIORITY_REFRESH,declining,0.893550,high_visibility_page,14.850833,0.034706,43321.0,0.012234,0.000234,landing
9,10,page_0025,PRIORITY_REFRESH,declining,0.893325,no_strong_signal,11.356000,0.019640,9641.0,0.015766,-0.001034,guide


## 6. Limitations & honest framing
- **Directional, not causal.** Nothing here proves *why* a page's ranking moved, and nothing here claims to reverse-engineer a search engine's ranking algorithm — these are decision-support signals, not causal claims.
- **Synthetic data stand-in.** Reported accuracy numbers describe how well the model recovers the synthetic simulation's own patterns, not real-world search performance. The pipeline, feature design, and validation methodology transfer directly to the real warehouse; the specific accuracy figures do not.
- **Class thresholds are a modeling choice.** The ±15% click-growth threshold for `growing`/`declining` is a reasonable default, not a law of nature — a content team should be able to tune it.
- **Out-of-time, not out-of-page.** The same 320 pages appear in both train and validation windows (at different times), so this validates generalization across *time*, not across entirely unseen pages. A production deployment should also periodically check performance on newly published pages the model has never scored before.

## 7. Reproducibility
- Weekly notebooks: `work/01_data_intake_and_eda.ipynb`, `work/02_duckdb_feature_aggregation.ipynb`, `work/03_baseline_and_model.ipynb`
- Shared library code: `src/data_gen.py`, `src/features.py`, `src/scoring_model.py`, `src/ranking_engine.py`, `src/make_figures.py`
- Regenerate everything end-to-end: `python src/data_gen.py && python src/make_figures.py`
- Deployed paper: see `submission/paper_url.txt`

## 8. Acknowledgments & data credit
Built on the FlyRank ML Internship dataset — [flyrank.ai](https://flyrank.ai) (this build substitutes a schema-matched synthetic dataset for the gated real warehouse; see Data section above).